In [19]:
write_mode     = "overwrite"

PM25_MAX   = 500.0   
NO2_MAX    = 2000.0  
O3_MAX     = 500.0   
VALUE_MIN  = 0.0     

BRONZE_BASE = (
    "abfss://itransition_de_project@onelake.dfs.fabric.microsoft.com"
    "/bronze.Lakehouse"
)

silver_table = "air_quality_silver"

StatementMeta(, e161972e-ce18-420a-9e03-76e5a29a4a64, 21, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, StringType, DateType

spark = SparkSession.builder.getOrCreate()

StatementMeta(, e161972e-ce18-420a-9e03-76e5a29a4a64, 4, Finished, Available, Finished, False)

In [8]:
raw_df = spark.read.format("delta").load(f"{BRONZE_BASE}/Tables/dbo/air_quality_raw")
raw_count = raw_df.count()

print(f"Bronze rows loaded : {raw_count:,}")
raw_df.printSchema()

StatementMeta(, e161972e-ce18-420a-9e03-76e5a29a4a64, 10, Finished, Available, Finished, False)

Bronze rows loaded : 341,706
root
 |-- _location_id: long (nullable = true)
 |-- _location_name: string (nullable = true)
 |-- _parameter: string (nullable = true)
 |-- _sensor_id: long (nullable = true)
 |-- _unit: string (nullable = true)
 |-- coordinates: string (nullable = true)
 |-- coverage: struct (nullable = true)
 |    |-- datetimeFrom: struct (nullable = true)
 |    |    |-- local: string (nullable = true)
 |    |    |-- utc: string (nullable = true)
 |    |-- datetimeTo: struct (nullable = true)
 |    |    |-- local: string (nullable = true)
 |    |    |-- utc: string (nullable = true)
 |    |-- expectedCount: long (nullable = true)
 |    |-- expectedInterval: string (nullable = true)
 |    |-- observedCount: long (nullable = true)
 |    |-- observedInterval: string (nullable = true)
 |    |-- percentComplete: double (nullable = true)
 |    |-- percentCoverage: double (nullable = true)
 |-- flagInfo: struct (nullable = true)
 |    |-- hasFlags: boolean (nullable = true)
 |--

In [9]:
from pyspark.sql import functions as F

silver_df = raw_df.select(
    F.col("_location_id").alias("location_id"),
    F.col("_location_name").alias("location_name"),
    F.col("_sensor_id").alias("sensor_id"),
    F.col("_parameter").alias("parameter"),
    F.col("_unit").alias("unit"),

    F.to_date(F.col("period.datetimeFrom.utc")).alias("measured_date"),
    F.col("period.datetimeFrom.utc").alias("period_from_utc"),
    F.col("period.datetimeTo.utc").alias("period_to_utc"),

    F.col("summary.avg").alias("value_avg"),
    F.col("summary.min").alias("value_min"),
    F.col("summary.max").alias("value_max"),
    F.col("summary.median").alias("value_median"),
    F.col("summary.sd").alias("value_sd"),

    F.col("coverage.percentComplete").alias("coverage_pct"),
    F.col("coverage.observedCount").alias("obs_count"),
    F.col("coverage.expectedCount").alias("expected_count"),
    F.col("flagInfo.hasFlags").alias("has_flags"),

    F.split(F.col("coordinates"), ",")[0].cast("double").alias("latitude"),
    F.split(F.col("coordinates"), ",")[1].cast("double").alias("longitude"),

    F.col("_ingested_at"),
)

silver_df = silver_df.filter(
    F.col("measured_date").isNotNull() &
    F.col("value_avg").isNotNull()
).dropDuplicates(["sensor_id", "measured_date", "parameter"])

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("air_quality_silver")
    
print(f"Silver rows : {silver_df.count():,}")
silver_df.printSchema()

StatementMeta(, e161972e-ce18-420a-9e03-76e5a29a4a64, 11, Finished, Available, Finished, False)

Silver rows : 68,896
root
 |-- location_id: long (nullable = true)
 |-- location_name: string (nullable = true)
 |-- sensor_id: long (nullable = true)
 |-- parameter: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- measured_date: date (nullable = true)
 |-- period_from_utc: string (nullable = true)
 |-- period_to_utc: string (nullable = true)
 |-- value_avg: double (nullable = true)
 |-- value_min: double (nullable = true)
 |-- value_max: double (nullable = true)
 |-- value_median: double (nullable = true)
 |-- value_sd: double (nullable = true)
 |-- coverage_pct: double (nullable = true)
 |-- obs_count: long (nullable = true)
 |-- expected_count: long (nullable = true)
 |-- has_flags: boolean (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)



In [15]:
filtered_df = silver_df.filter(F.col("value_avg") >= VALUE_MIN) \
    .filter(F.col("parameter").isin("pm25", "no2", "o3")) \
    .filter(
        F.when(F.col("parameter") == "pm25", F.col("value_avg") <= PM25_MAX)
         .when(F.col("parameter") == "no2",  F.col("value_avg") <= NO2_MAX)
         .when(F.col("parameter") == "o3",   F.col("value_avg") <= O3_MAX)
         .otherwise(True)
    )

StatementMeta(, e161972e-ce18-420a-9e03-76e5a29a4a64, 17, Finished, Available, Finished, False)

In [16]:
daily_avg_df = (
    filtered_df
    .groupBy("location_id", "location_name", "latitude", "longitude", "measured_date")
    .pivot("parameter", ["pm25", "no2", "o3"])
    .agg(F.round(F.avg("value_avg"), 4))
)

pivot_df = daily_avg_df \
    .withColumnRenamed("pm25", "avg_pm25") \
    .withColumnRenamed("no2",  "avg_no2") \
    .withColumnRenamed("o3",   "avg_o3")

clean_df = pivot_df.filter(
    F.col("avg_pm25").isNotNull() |
    F.col("avg_no2").isNotNull()  |
    F.col("avg_o3").isNotNull()
)

StatementMeta(, e161972e-ce18-420a-9e03-76e5a29a4a64, 18, Finished, Available, Finished, False)

In [17]:
enriched_df = clean_df.withColumn(
    "location_key",
    F.sha2(
        F.concat_ws("|",
            F.col("location_id").cast(StringType()),
            F.coalesce(F.col("location_name"), F.lit("")),
        ),
        256
    )
)

StatementMeta(, e161972e-ce18-420a-9e03-76e5a29a4a64, 19, Finished, Available, Finished, False)

In [20]:
silver_count = enriched_df.count()
print(f"\nRows written to silver : {silver_count:,}")

(
    enriched_df.write
    .format("delta")
    .mode(write_mode)
    .partitionBy("measured_date")          # was measurement_date
    .option("overwriteSchema", "true")
    .saveAsTable(silver_table)
)

print(f"[OK] silver.{silver_table} written  (mode={write_mode})")

StatementMeta(, e161972e-ce18-420a-9e03-76e5a29a4a64, 22, Finished, Available, Finished, False)


Rows written to silver : 49,812
[OK] silver.air_quality_silver written  (mode=overwrite)


In [22]:
spark.sql(f"""
    SELECT
        MIN(measured_date)          AS earliest,
        MAX(measured_date)          AS latest,
        COUNT(*)                    AS location_day_rows,
        COUNT(DISTINCT location_id) AS unique_locations,
        ROUND(AVG(avg_pm25), 2)     AS mean_pm25,
        ROUND(AVG(avg_no2),  2)     AS mean_no2,
        ROUND(AVG(avg_o3),   2)     AS mean_o3,
        SUM(CASE WHEN avg_pm25 IS NULL THEN 1 ELSE 0 END) AS missing_pm25,
        SUM(CASE WHEN avg_no2  IS NULL THEN 1 ELSE 0 END) AS missing_no2,
        SUM(CASE WHEN avg_o3   IS NULL THEN 1 ELSE 0 END) AS missing_o3
    FROM {silver_table}
""").show()

StatementMeta(, e161972e-ce18-420a-9e03-76e5a29a4a64, 24, Finished, Available, Finished, False)

+----------+----------+-----------------+----------------+---------+--------+-------+------------+-----------+----------+
|  earliest|    latest|location_day_rows|unique_locations|mean_pm25|mean_no2|mean_o3|missing_pm25|missing_no2|missing_o3|
+----------+----------+-----------------+----------------+---------+--------+-------+------------+-----------+----------+
|2016-03-06|2026-05-21|            49812|              54|     8.28|    0.02|   0.03|        1893|      40552|     38096|
+----------+----------+-----------------+----------------+---------+--------+-------+------------+-----------+----------+

